# Sequence-similarity invisible AVGs

Characterize the de-novo AVGs that no `annotate` database hits: how many DefenseFinder and structural search identify, how they group into embedding-based families, and how deep a propagated label reaches for those no method identifies. Propagated labels are read from `tables/propagation/protein_assignments.parquet`. This notebook assigns none.

- **De-novo-only AVG.** A protein called an AVG by `de-novo` at medium or higher confidence that `annotate` did not call an AVG.
- **Sequence-similarity invisible AVG.** A de-novo-only AVG with no `annotate` HMM hit, meaning invisible to the seven `annotate` databases, not without relatives.
- **Unresolved AVG.** A sequence-similarity invisible AVG that neither DefenseFinder nor structural search identifies.

## Setup

Imports and thread configuration.

In [1]:
import os
os.environ["POLARS_MAX_THREADS"] = "48"
import glob
import json
import time
import re
from pathlib import Path
import numpy as np
import polars as pl
import faiss
import tables as tb
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(20)
N_THREADS = 48
faiss.omp_set_num_threads(N_THREADS)
SEED = 20260715
rng = np.random.default_rng(SEED)
CAT3 = ["metabolic", "physiological", "regulatory"]

Paths.

In [2]:
REPO = Path("/storage2/scratch/kosmopoulos/software/CheckAMG")
FILES = REPO / "CheckAMG" / "files"
TBL_DIR = REPO / "notebooks" / "tables"
PROP = TBL_DIR / "propagation"
FIGT = TBL_DIR / "novel_avgs"
FIGT.mkdir(parents=True, exist_ok=True)
MAIN = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript")
OUT = MAIN / "novel_avgs"
OUT.mkdir(parents=True, exist_ok=True)
DENOVO_DB = REPO / "notebooks" / "CheckAMG_denovo_db_v1.1_20260714"
DB_STAMP = DENOVO_DB.name.replace("CheckAMG_denovo_db_", "")
PCACHE = MAIN / "propagation" / DB_STAMP
DATASETS = ["soil", "gut"]
def den(ds):
    return MAIN / f"CheckAMG_denovo_v1.1_MetaVR{ds}"
DF_DB_DIR = Path("/storage2/scratch/kosmopoulos/databases/defensefinder/defense-finder-models")
DF_PROFILE_DIR = DF_DB_DIR / "profiles"
CHECKAMG_ALL_HMMS_PATH = FILES / "hmm_id_to_name.tsv"
ENVS = Path("/storage2/scratch/kosmopoulos/miniconda3/envs")
print("OUT:", OUT)

OUT: /storage2/scratch/kosmopoulos/projects/checkAMG/applied_runs_for_manuscript/novel_avgs


Load the canonical assignments, assert the tiered-propagation schema version, and report which tier produced each label over the sequence-similarity invisible set.


In [3]:
SCHEMA_VERSION = "propagation-1.2"
manifest = json.loads((PROP / "manifest.json").read_text())
assert manifest["schema_version"] == SCHEMA_VERSION, \
    f"propagation schema {manifest['schema_version']} != expected {SCHEMA_VERSION}; rerun functional_propagation.ipynb"
assert manifest["label_source_vocabulary"] == ["cluster", "embedding", "none"], \
    f"unexpected label_source vocabulary {manifest['label_source_vocabulary']}"
assign = pl.read_parquet(PROP / "protein_assignments.parquet")
TIER_COLS = ["label_source", "tier1_label", "tier1_level", "tier2_label", "tier2_level", "cluster_reachable"]
_absent = [c for c in TIER_COLS if c not in assign.columns]
assert not _absent, f"tiered columns absent from protein_assignments.parquet: {_absent}"
hi = assign.filter(pl.col("sequence_similarity_invisible"))
print(f"assignments {assign.height:,} rows x {assign.width} columns "
      f"(schema {manifest['schema_version']}, target {manifest['primary_target']:.0%})")
print(f"de-novo-only AVGs      : {assign.filter(pl.col('module') == 'de_novo').height:,}")
print(f"sequence-similarity invisible AVGs: {hi.height:,}")
print(hi.group_by("final_level").agg(pl.len().alias("n"))
        .with_columns((100 * pl.col("n") / hi.height).round(1).alias("pct")).sort("n", descending=True))
print(f"labeled at the target: {hi.filter(pl.col('final_level') != 'unassigned').height:,}")
print(hi.group_by("label_source").agg(pl.len().alias("n"))
        .with_columns((100 * pl.col("n") / hi.height).round(1).alias("pct")).sort("n", descending=True))
_reach = int(hi["cluster_reachable"].sum())
print(f"present in the MMseqs2 clustering universe: {_reach:,} "
      f"({100 * _reach / hi.height:.1f}%)")
for _nm, _c in [("tier 1 (cluster majority)", "tier1_level"), ("tier 2 (embedding 1-NN)", "tier2_level")]:
    _k = hi.filter(pl.col(_c) != "unassigned").height
    print(f"{_nm:26s} alone reaches {_k:,} ({100 * _k / hi.height:.1f}%)")


assignments 541,199 rows x 43 columns (schema propagation-1.2, target 90%)
de-novo-only AVGs      : 189,385
sequence-similarity invisible AVGs: 109,107
shape: (4, 3)
┌─────────────┬───────┬──────┐
│ final_level ┆ n     ┆ pct  │
│ ---         ┆ ---   ┆ ---  │
│ str         ┆ u32   ┆ f64  │
╞═════════════╪═══════╪══════╡
│ specific    ┆ 45392 ┆ 41.6 │
│ unassigned  ┆ 37957 ┆ 34.8 │
│ L1          ┆ 20651 ┆ 18.9 │
│ category    ┆ 5107  ┆ 4.7  │
└─────────────┴───────┴──────┘
labeled at the target: 71,150
shape: (3, 3)
┌──────────────┬───────┬──────┐
│ label_source ┆ n     ┆ pct  │
│ ---          ┆ ---   ┆ ---  │
│ str          ┆ u32   ┆ f64  │
╞══════════════╪═══════╪══════╡
│ embedding    ┆ 44606 ┆ 40.9 │
│ none         ┆ 37957 ┆ 34.8 │
│ cluster      ┆ 26544 ┆ 24.3 │
└──────────────┴───────┴──────┘
present in the MMseqs2 clustering universe: 109,107 (100.0%)
tier 1 (cluster majority)  alone reaches 35,242 (32.3%)


tier 2 (embedding 1-NN)    alone reaches 58,471 (53.6%)


## Step 1: Verify held-out status of the DefenseFinder label sources

For every DefenseFinder and AntiDefenseFinder subtype, determine whether any constituent profile has a recoverable equivalent in the seven CheckAMG databases, assessed by profile accession overlap and by normalized description match. Systems that pass both are the held-out label set.

Parse the searchable DefenseFinder profile HMMs.

In [4]:
def parse_hmm(hmm_path):
    d = {"gene_name": hmm_path.stem, "hmm_NAME": None, "hmm_ACC": None, "hmm_GA": None}
    with open(hmm_path) as f:
        for raw in f:
            if raw.startswith("NAME"):
                d["hmm_NAME"] = raw.split()[1]
            elif raw.startswith("ACC"):
                d["hmm_ACC"] = raw.split()[1]
            elif raw.startswith("GA"):
                p = raw.replace(";", " ").split()
                d["hmm_GA"] = float(p[1]) if len(p) > 1 else None
            elif raw.startswith("CKSUM"):
                break
    return d
df_profiles = pl.from_dicts(
    [parse_hmm(Path(p)) for p in sorted(glob.glob(str(DF_PROFILE_DIR / "*.hmm")))],
    schema={"gene_name": pl.String, "hmm_NAME": pl.String, "hmm_ACC": pl.String, "hmm_GA": pl.Float64})
print(f"searchable DefenseFinder profiles: {df_profiles.height:,}")

searchable DefenseFinder profiles: 1,332


Parse the profile to system metadata table and reconcile it with the profiles.

In [5]:
rows = []
for line in (DF_DB_DIR / "Liste_hmm_system.md").read_text().splitlines():
    if line.strip().startswith("|"):
        parts = [c.strip() for c in line.strip().strip("|").split("|")]
        if len(parts) >= 5:
            rows.append(parts[:5])
df_meta = pl.DataFrame({rows[0][i]: [r[i] for r in rows[2:]] for i in range(5)})
df_meta = (df_meta.with_columns([pl.when(pl.col(c) == "").then(None).otherwise(pl.col(c)).alias(c)
                                 for c in df_meta.columns])
           .rename({"System": "meta_System", "Accession": "meta_Accession",
                    "GA_cut": "meta_GA", "HMM_name": "meta_HMM_name"})
           .with_columns(pl.col("meta_GA").cast(pl.Float64)))
df_profiles = (df_profiles.join(df_meta.select("gene_name", "meta_System", "meta_Accession"),
                                on="gene_name", how="left")
               .with_columns(pl.coalesce([pl.col("meta_System"),
                                          pl.col("gene_name").str.split("__").list.first()]).alias("System"),
                             pl.coalesce([pl.col("meta_Accession"), pl.col("hmm_ACC")]).alias("Accession"),
                             pl.col("meta_System").is_null().alias("system_from_prefix")))
print(f"profiles resolved to a System: {df_profiles.filter(pl.col('System').is_not_null()).height:,}"
      f"/{df_profiles.height:,} ({df_profiles['System'].n_unique()} distinct systems)")

profiles resolved to a System: 1,332/1,332 (598 distinct systems)


Accession-level and description-level crosswalk against the seven CheckAMG databases. A system counts as held out if it shares neither a Pfam accession nor a function-name token with those databases.

In [6]:
checkamg_hmms = pl.read_csv(CHECKAMG_ALL_HMMS_PATH, separator="\t")
ca_pfam = set(checkamg_hmms.filter(pl.col("db") == "Pfam")["id"].str.replace(r"\..*", "").to_list())
df_profiles = (df_profiles.with_columns(
    pl.when(pl.col("Accession").str.starts_with("PF"))
      .then(pl.col("Accession").str.replace(r"\..*", "")).otherwise(None).alias("pfam"))
    .with_columns(pl.col("pfam").is_in(list(ca_pfam)).alias("pfam_in_checkamg")))
df_systems = (df_profiles.group_by("System").agg(
        pl.len().alias("n_profiles"),
        pl.col("pfam").is_not_null().any().alias("any_pfam"),
        pl.col("pfam_in_checkamg").any().alias("any_pfam_in_checkamg"))
    .with_columns(pl.when(pl.col("any_pfam_in_checkamg")).then(pl.lit("accession_visible"))
                    .otherwise(pl.lit("accession_held_out")).alias("accession_visibility")).sort("System"))
GEN = {"protein", "family", "domain", "putative", "system", "type", "like", "containing",
       "uncharacterized", "group", "associated", "related"}
def toks(s):
    return {w for w in re.findall(r"[a-z0-9]{4,}", str(s).lower()) if w not in GEN}
ca_tokens = set()
for n in checkamg_hmms["name"].to_list():
    ca_tokens |= toks(n)
held = set(df_systems.filter(pl.col("accession_visibility") == "accession_held_out")["System"].to_list())
gene_by_sys = {s: g for s, g in df_profiles.group_by("System").agg(pl.col("gene_name")).iter_rows()}
desc_visible = []
for s in held:
    st = toks(s)
    for g in gene_by_sys.get(s, []):
        st |= toks(g)
    if st & ca_tokens:
        desc_visible.append(s)
robust = held - set(desc_visible)
df_systems = df_systems.with_columns(
    pl.col("System").is_in(desc_visible).alias("name_token_in_checkamg"),
    pl.when(pl.col("accession_visibility") == "accession_visible").then(pl.lit("visible"))
      .when(pl.col("System").is_in(list(robust))).then(pl.lit("held_out"))
      .otherwise(pl.lit("description_visible")).alias("visibility_refined"))
df_profiles.write_parquet(OUT / "defensefinder_profiles.parquet")
df_systems.write_parquet(OUT / "defensefinder_systems.parquet")
print(f"accession-level held-out: {len(held):,}; of which name-visible {len(desc_visible):,}; "
      f"held out by accession and name {len(robust):,}")
print(df_systems.group_by("visibility_refined").len().sort("len", descending=True))

accession-level held-out: 584; of which name-visible 163; held out by accession and name 421
shape: (3, 2)
┌─────────────────────┬─────┐
│ visibility_refined  ┆ len │
│ ---                 ┆ --- │
│ str                 ┆ u32 │
╞═════════════════════╪═════╡
│ held_out            ┆ 421 │
│ description_visible ┆ 163 │
│ visible             ┆ 14  │
└─────────────────────┴─────┘


## Step 2: DefenseFinder recovery of the sequence-similarity invisible AVGs

The per-profile scan runs in `functional_propagation.ipynb`, because it depends only on the sequence-similarity invisible set that notebook defines. This step reads its output.

Read the DefenseFinder recovery partition.

In [7]:
df_hits = pl.read_parquet(PCACHE / "defensefinder" / "hi_defensefinder_hits.parquet")
df_recovery = pl.read_parquet(PCACHE / "defensefinder" / "hi_df_recovery.parquet")
N = df_recovery.height
print(f"sequence-similarity invisible AVGs ({N:,}) by DefenseFinder recovery:")
for r in df_recovery.group_by("df_recovery").len().sort("len", descending=True).iter_rows(named=True):
    print(f"  {r['df_recovery']:24s} {r['len']:>8,} ({100 * r['len'] / N:.1f}%)")
print("\ntop DefenseFinder systems recovered at GA:")
print(df_hits.filter(pl.col("ga_pass")).group_by("System")
      .agg(pl.col("Protein").n_unique().alias("n_proteins")).sort("n_proteins", descending=True).head(15))

sequence-similarity invisible AVGs (109,107) by DefenseFinder recovery:
  no_hit_P_novel             85,810 (78.6%)
  recovered_GA               14,778 (13.5%)
  recovered_relaxed_only      8,519 (7.8%)

top DefenseFinder systems recovered at GA:
shape: (15, 2)
┌────────────────┬────────────┐
│ System         ┆ n_proteins │
│ ---            ┆ ---        │
│ str            ┆ u32        │
╞════════════════╪════════════╡
│ RosmerTA       ┆ 12231      │
│ MADS           ┆ 419        │
│ RM_Type_II     ┆ 374        │
│ Armada_type_I  ┆ 231        │
│ AbiR           ┆ 210        │
│ RM             ┆ 125        │
│ Armada_type_II ┆ 87         │
│ AbiEii         ┆ 78         │
│ Gao_Iet        ┆ 72         │
│ CBASS          ┆ 71         │
│ Dnd            ┆ 67         │
│ Gao_Qat        ┆ 66         │
│ BREX           ┆ 66         │
│ Pif            ┆ 65         │
│ Zorya_TypeI    ┆ 63         │
└────────────────┴────────────┘


## Step 2b: Annotation status of the training homologs

Sequence-similarity invisible means no hit to the seven `annotate` databases, not an absence of relatives in the CheckAMG-PST training data. This step measures how many of those relatives carry a functional annotation. Identity comes from the MMseqs2 search of every sequence-similarity invisible AVG against all training proteins, cached by functional_propagation.ipynb, which is separate from the clustering that tier 1 votes over.

Fraction of matched training proteins carrying a real function, by identity band.

In [8]:
UNKNOWN = (r"(?i)hypothetical|uncharacteri[sz]ed|unknown function|DUF[0-9]|domain of unknown|"
           r"unnamed|predicted protein|protein of unknown")
TRAIN_ANNO = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1")
ann = pl.concat([pl.read_parquet(TRAIN_ANNO / f"checkamg_annotate_{d}_dataset" / "results" / "final_results.parquet",
                                 columns=["Protein", "Protein Classification", "Function"])
                 for d in ["genomad", "progenomes"]]).unique("Protein")
ann = ann.with_columns(
    (pl.col("Function").is_not_null() & (pl.col("Function") != "")
     & ~pl.col("Function").str.contains(UNKNOWN)).alias("has_real_function"),
    pl.col("Protein Classification").is_in(CAT3).alias("is_avglike"))
BASE_LABELLED = float(ann["has_real_function"].mean())
print(f"training proteins: {ann.height:,}; carrying a real function: "
      f"{int(ann['has_real_function'].sum()):,} ({100*BASE_LABELLED:.1f}%)")
aln = pl.read_csv(PCACHE / "hi_vs_train_alignments.tsv", separator="\t", has_header=False,
                  new_columns=["query", "target", "fident", "evalue", "bits", "qcov", "tcov"])
aln = (aln.join(ann.rename({"Protein": "target"}), on="target", how="left")
          .with_columns(pl.col("has_real_function").fill_null(False), pl.col("is_avglike").fill_null(False)))
BANDS = [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.0]
rows = []
for t in BANDS:
    s = aln.filter(pl.col("fident") >= t)
    per = s.group_by("query").agg(pl.col("has_real_function").any().alias("lab"),
                                  pl.col("is_avglike").any().alias("avg"))
    tp = s.unique("target")
    rows.append({"min_identity": t, "hi_avgs_with_hit": per.height,
                 "pct_of_all_hi": 100 * per.height / N,
                 "hi_with_labeled_homolog": int(per["lab"].sum()),
                 "pct_of_band_labeled": 100 * float(per["lab"].mean()) if per.height else 0.0,
                 "pct_of_all_hi_labeled": 100 * int(per["lab"].sum()) / N,
                 "hi_with_avglike_homolog": int(per["avg"].sum()),
                 "distinct_targets": tp.height,
                 "pct_targets_labeled": 100 * float(tp["has_real_function"].mean()) if tp.height else 0.0})
hl = pl.DataFrame(rows)
hl.write_parquet(FIGT / "fig_hi_training_homolog_labels.parquet")
print(hl.select(["min_identity", "hi_avgs_with_hit", "pct_of_all_hi", "hi_with_labeled_homolog",
                 "pct_of_all_hi_labeled", "pct_targets_labeled"]))
print(f"\nno training alignment at all: {N - hl['hi_avgs_with_hit'].max():,} "
      f"({100*(N - hl['hi_avgs_with_hit'].max())/N:.1f}%)")

training proteins: 16,825,811; carrying a real function: 7,660,167 (45.5%)


shape: (9, 6)
┌──────────────┬────────────────┬───────────────┬────────────────┬────────────────┬────────────────┐
│ min_identity ┆ hi_avgs_with_h ┆ pct_of_all_hi ┆ hi_with_labele ┆ pct_of_all_hi_ ┆ pct_targets_la │
│ ---          ┆ it             ┆ ---           ┆ d_homolog      ┆ labeled        ┆ beled          │
│ f64          ┆ ---            ┆ f64           ┆ ---            ┆ ---            ┆ ---            │
│              ┆ i64            ┆               ┆ i64            ┆ f64            ┆ f64            │
╞══════════════╪════════════════╪═══════════════╪════════════════╪════════════════╪════════════════╡
│ 0.9          ┆ 24533          ┆ 22.485267     ┆ 3278           ┆ 3.00439        ┆ 20.946346      │
│ 0.8          ┆ 31678          ┆ 29.033884     ┆ 6357           ┆ 5.826391       ┆ 25.278509      │
│ 0.7          ┆ 39130          ┆ 35.863877     ┆ 11639          ┆ 10.66751       ┆ 30.024021      │
│ 0.6          ┆ 51816          ┆ 47.490995     ┆ 20659          ┆ 18.934624 

## Step 3: Structural search

Structural homology is orthogonal to sequence databases and to the model, so it provides independent evidence that a functional assignment corresponds to a real function rather than an embedding artifact. Foldseek runs on CPU in ProstT5 mode, which predicts 3Di structural tokens from sequence without explicit folding.

Each database search is cached to save runtime and skipped when its alignment file exists.

Tool and database provenance, captured at run time.

In [9]:
import subprocess
FOLDSEEK = ENVS / "pholdENV" / "bin" / "foldseek"
PROSTT5_MODEL = Path("/storage2/scratch/kosmopoulos/databases/foldseek_prostt5_v10/prostt5-f16.gguf")
_FSDB = Path("/storage2/databases/foldseek")
TARGETS = {"pdb": _FSDB / "pdb100" / "pdb", "bfvd": _FSDB / "BFVD" / "db",
           "afdb50": _FSDB / "Alphafold" / "UniProt50" / "afdb"}
def _rd(p):
    try:
        return Path(p).read_text().strip().replace("\n", " | ")
    except Exception:
        return "n/a"
STEP3_PROVENANCE = {
    "foldseek": subprocess.run([str(FOLDSEEK), "version"], capture_output=True, text=True).stdout.strip(),
    "ProstT5": "Rostlab/ProstT5 (foldseek prostt5-f16 weights); Heinzinger et al. 2024",
    "PDB (foldseek pdb100)": _rd(_FSDB / "pdb100" / "pdb.version"),
    "BFVD": _rd(_FSDB / "BFVD" / "db.version"),
    "AFDB50": "AlphaFold DB via foldseek databases",
    "target databases searched": ["PDB", "BFVD", "AFDB50"]}
for k, v in STEP3_PROVENANCE.items():
    print(f"  {k}: {v}")
FAA = PCACHE / "defensefinder" / "homology_invisible.faa"
E_STRONG, QCOV_STRONG = 1e-5, 0.5
UNLAB = re.compile(r"(?i)hypothetical|uncharacteri[sz]ed|unknown function|DUF[0-9]|domain of unknown|"
                   r"unnamed|putative uncharacter|predicted protein|protein of unknown")

  foldseek: 10.941cd33
  ProstT5: Rostlab/ProstT5 (foldseek prostt5-f16 weights); Heinzinger et al. 2024
  PDB (foldseek pdb100): df8de2b01408fe438e3f075698032906  pdb100.tar.gz | 240101	PDB_DATE | b6dac8a54139632c9d251ebd3540df05fcd9bc54	FOLDSEEK_COMMIT
  BFVD: 2023_02_v2
  AFDB50: AlphaFold DB via foldseek databases
  target databases searched: ['PDB', 'BFVD', 'AFDB50']


## Step 4: Cluster the sequence-similarity invisible AVGs in embedding space

All sequence-similarity invisible AVGs are clustered on a cosine kNN graph over CheckAMG-PST embeddings with Leiden community detection. Clusters are the protein families used throughout the novelty analysis.

**Parameters.** Embeddings are L2-normalized. The kNN graph is built with a FAISS HNSW inner-product index, `M` 32, `efConstruction` 200, `efSearch` 64, keeping the 15 nearest neighbors excluding self, and edges are added undirected and deduplicated. Communities come from `igraph` Leiden with `objective_function` modularity and `n_iterations` 5. The HNSW graph is built single-threaded and Leiden is seeded, so a rebuild is deterministic, and the membership is cached to save runtime.

Extract embeddings, build the kNN graph, and run Leiden.

In [10]:
import random
import igraph as ig
EMB, KEYS, MEMB = OUT / "hi_emb.npy", OUT / "hi_emb_keys.parquet", OUT / "hi_cluster_membership.npy"
if EMB.exists() and KEYS.exists():
    emb = np.load(EMB)
    keys = pl.read_parquet(KEYS)
    print("loaded cached hi embeddings", emb.shape)
else:
    t0 = time.time()
    ec, kc = [], []
    for ds in DATASETS:
        pred = pl.read_csv(den(ds) / "predictions.tsv", separator="\t", columns=["Protein"]).with_row_index("row")
        want = pred.join(hi.filter(pl.col("dataset") == ds).select(pl.col("protein_id").alias("Protein")),
                         on="Protein", how="inner").sort("row")
        m = np.zeros(pred.height, bool)
        m[want["row"].to_numpy()] = True
        buf = []
        with tb.open_file(str(den(ds) / "combined_proteins.filtered.PST-EMBED.h5")) as fp:
            a = fp.root.ctx_ptn
            n = a.shape[0]
            for s in range(0, n, 500_000):
                e = min(s + 500_000, n)
                mm = m[s:e]
                if mm.any():
                    buf.append(a[s:e][mm])
        ec.append(np.concatenate(buf).astype(np.float32))
        kc.append(want.select(["Protein", pl.lit(ds).alias("ecosystem")]))
    emb = np.concatenate(ec)
    keys = pl.concat(kc)
    np.save(EMB, emb)
    keys.write_parquet(KEYS)
    print(f"extracted hi embeddings {emb.shape} ({time.time()-t0:.0f}s)")
if MEMB.exists():
    cl = np.load(MEMB)
    print(f"loaded cached Leiden membership: {cl.max()+1} clusters")
else:
    t0 = time.time()
    en = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-8, None)
    idx = faiss.IndexHNSWFlat(en.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = 200
    # Single-threaded insertion makes the HNSW graph, and so the kNN graph, reproducible
    faiss.omp_set_num_threads(1)
    idx.add(en)
    faiss.omp_set_num_threads(N_THREADS)
    idx.hnsw.efSearch = 64
    _, nbr = idx.search(en, 16)
    edges = []
    for i in range(en.shape[0]):
        for j in nbr[i, 1:]:
            if j >= 0 and i < j:
                edges.append((i, int(j)))
    g = ig.Graph(n=en.shape[0], edges=edges)
    g.simplify()
    # igraph draws its random numbers from Python's random module
    random.seed(SEED)
    cl = np.array(g.community_leiden(objective_function="modularity", n_iterations=5).membership)
    np.save(MEMB, cl)
    print(f"Leiden: {cl.max()+1} clusters ({time.time()-t0:.0f}s)")
tab = (keys.with_columns(pl.Series("cluster", cl))
       .join(assign.select([pl.col("protein_id").alias("Protein"), "final_label", "final_level",
                            "raw_category", "raw_L1", "d1", "p_specific", "p_L1", "p_category",
                            "label_source", "tier1_level", "tier2_level"]),
             on="Protein", how="left")
       .join(df_recovery.select(["Protein", "df_recovery"]), on="Protein", how="left"))
print(f"clustered proteins: {tab.height:,} in {tab['cluster'].n_unique():,} clusters")

loaded cached hi embeddings (109107, 800)
loaded cached Leiden membership: 291 clusters
clustered proteins: 109,107 in 291 clusters


Foldseek search of every sequence-similarity invisible AVG against PDB, BFVD, and AFDB50. The queries are split into 24 chunks searched in parallel with 4 threads each. AFDB50 is split in memory and searched 12 chunks at a time so that it fits in RAM.

In [ ]:
import shutil
S3 = OUT / "step3_all"
M8 = [S3 / f"all_vs_{db}.m8" for db in ["pdb", "bfvd", "afdb50"]]
N_CHUNKS = 24

def foldseek_search(db):
    extra, max_parallel = (["--split-memory-limit", "100G"], 12) if db == "afdb50" else ([], N_CHUNKS)
    running = []
    for chunk in sorted((S3 / "chunks").glob("q*.faa")):
        m8 = S3 / f"{chunk.stem}_vs_{db}.m8"
        if m8.exists() and m8.stat().st_size > 0:
            continue
        while sum(p.poll() is None for p in running) >= max_parallel:
            time.sleep(20)
        with open(S3 / f"log_{chunk.stem}_{db}.txt", "w") as log:
            running.append(subprocess.Popen(
                [str(FOLDSEEK), "easy-search", str(chunk), str(TARGETS[db]), str(m8), str(S3 / f"tmp_{chunk.stem}_{db}"),
                 "--prostt5-model", str(PROSTT5_MODEL), "--threads", "4", "-e", "0.01", "-s", "9.5", "--max-seqs", "300",
                 *extra, "--format-output", "query,target,theader,fident,alnlen,evalue,bits,qcov,tcov"],
                stdout=log, stderr=subprocess.STDOUT))
    assert all(p.wait() == 0 for p in running), f"a Foldseek search against {db} failed"
    with open(S3 / f"all_vs_{db}.m8", "wb") as out:
        for part in sorted(S3.glob(f"q*_vs_{db}.m8")):
            with open(part, "rb") as fh:
                shutil.copyfileobj(fh, out)

if not all(f.exists() and f.stat().st_size > 0 for f in M8):
    (S3 / "chunks").mkdir(parents=True, exist_ok=True)
    if not (S3 / "chunks" / "q00.faa").exists():
        chunk_files = [open(S3 / "chunks" / f"q{i:02d}.faa", "w") for i in range(N_CHUNKS)]
        record = -1
        with open(FAA) as fh:
            for line in fh:
                if line[0] == ">":
                    record += 1
                chunk_files[record % N_CHUNKS].write(line)
        for chunk_file in chunk_files:
            chunk_file.close()
    for db in ["pdb", "bfvd", "afdb50"]:
        if not (S3 / f"all_vs_{db}.m8").exists():
            foldseek_search(db)
print("per-protein structural search inputs:")
for f in M8:
    print(f"  {f.name}: {f.stat().st_size/1e9:.2f} GB")

per-protein structural search inputs:
  all_vs_pdb.m8: 0.84 GB
  all_vs_bfvd.m8: 0.63 GB
  all_vs_afdb50.m8: 4.77 GB


Resolve a structural tier and theme per protein. A protein is `Structure: known function` if its own best confident alignment names a specific function, `Structure: unknown function` if it aligns but only to uncharacterized or domain-level targets, and `No structural match` if it has no alignment at all.

BFVD target headers carry only a UniProt accession, so a BFVD hit has no description of its own. Where a hit has no description, the accession's UniProt protein name is used instead, and it competes on bit score under the same best-hit rule as the other databases. The name lookup streams the full UniProt name tables once and is cached.

In [12]:
BFVD_NAMES_PATH = PCACHE / "bfvd_accession_names.parquet"

if not BFVD_NAMES_PATH.exists():
    UNIPROT_TABLES = Path("/storage2/databases/MicrobeAnnotator_DB")
    _want = set()
    with open(S3 / "all_vs_bfvd.m8", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            p = line.rstrip("\n").split("\t")
            if len(p) < 8:
                continue
            if float(p[-4]) <= E_STRONG and float(p[-2]) >= QCOV_STRONG:
                _want.add(p[1])
    _name = {}
    for _f in ["uniprot_swissprot.table", "uniprot_trembl.table"]:
        with open(UNIPROT_TABLES / _f, encoding="utf-8", errors="replace") as fh:
            for line in fh:
                q = line.split("\t", 3)
                if len(q) < 3:
                    continue
                if q[1] in _want and q[1] not in _name:
                    _name[q[1]] = q[2]
    (pl.DataFrame({"accession": list(_name.keys()), "protein_name": list(_name.values())})
       .with_columns(pl.col("protein_name").str.replace_all(r"\s+", " ").str.strip_chars())
       .write_parquet(BFVD_NAMES_PATH))
    print(f"built the BFVD name cache: {len(_name):,} of {len(_want):,} accessions resolved")

_bn = pl.read_parquet(BFVD_NAMES_PATH)
BFVD_NAMES = dict(zip(_bn["accession"].to_list(), _bn["protein_name"].to_list()))
print(f"BFVD accession names available: {len(BFVD_NAMES):,}")

BFVD accession names available: 20,865


In [13]:
AUTO = re.compile(r"(?i)domain[- ]containing|-like protein$")
JUNKD = re.compile(r"(?i)whole genome shotgun|^contig|^scaffold|genome assembly|^chromosome\b|genomic scaffold")
BOILER = re.compile(r"(?i)^(crystal |solution |cryo-?em |nmr )?structure of (the )?|^crystal structure,?\s*")
ORGCL = re.compile(r"(?i)\s+(from|of)\s+[A-Z][a-z]+\s+[a-z]{3,}.*$")
CPLX = re.compile(r"(?i)\s+(bound to|complexed (to|with)|in complex with).*$")
def dsc(h):
    p = h.split(" ", 1)
    d = p[1] if len(p) > 1 else ""
    d = re.sub(r"mol:\w+\s+length:\d+\s*", "", d)
    d = BOILER.sub("", d)
    d = CPLX.sub("", d)
    d = ORGCL.sub("", d)
    return re.sub(r"\s+", " ", d).strip()
THEMES = [("Physiological", r"(?i)anti-?crispr|anti-?restriction|restriction|crispr|cas\d|toxin|antitoxin|abortive|thoeris|cbass|gabija|retron|defen|immunit|nuclease"),
          ("Regulatory", r"(?i)\bhth\b|helix-turn-helix|transcription|repressor|activator|sigma|luxr|\bcro\b|\bxre\b|regulator|dna[- ]binding"),
          ("Other", r"(?i)parb|repl|primase|helicase|recombinase|integrase|resolvase|transposase|bro-n|single-strand|topoisomer"),
          ("Other", r"(?i)capsid|tail|portal|terminase|baseplate|\bneck\b|sheath|fiber|head"),
          ("Metabolic", r"(?i)hydrolase|transferase|kinase|peptidase|synthase|synthetase|dehydrogenase|reductase|lyase|esterase|oxidase|phosphatase|glycosyl")]
def theme(d):
    for nm, pat in THEMES:
        if re.search(pat, d):
            return nm
    return "Other"

# Description regexes are the expensive step, so they run only on hits that pass the e-value and coverage thresholds
t0 = time.time()
best, has_any, nlines = {}, set(), 0
for db, f in zip(["pdb", "bfvd", "afdb50"], M8):
    with open(f, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            p = line.rstrip("\n").split("\t")
            if len(p) < 8:
                continue
            nlines += 1
            q = p[0]
            has_any.add(q)
            ev, bits, qc = float(p[-4]), float(p[-3]), float(p[-2])
            if not (ev <= E_STRONG and qc >= QCOV_STRONG):
                continue
            d = dsc("\t".join(p[2:-6]))
            from_bfvd_names = False
            if not d:
                d = BFVD_NAMES.get(p[1], "")
                from_bfvd_names = bool(d)
            named = bool(d) and len(d) > 3 and not UNLAB.search(d) and not JUNKD.search(d)
            if not named:
                continue
            tier = 2 if not AUTO.search(d) else 1
            key = (tier, bits)
            if q not in best or key > best[q][0]:
                best[q] = (key, tier, d, qc, db, p[1], from_bfvd_names, bits, ev)
print(f"parsed {nlines:,} alignments in {time.time()-t0:.0f}s")
print(f"proteins with any hit: {len(has_any):,}   with a confident named hit: {len(best):,}")

# Each protein is classified on its own alignments
_prot = tab["Protein"].to_list()
_tier, _theme, _qcov = [], [], []
for q in _prot:
    b = best.get(q)
    if b is not None and b[1] == 2:
        _tier.append("Structure: known function")
        _theme.append(theme(b[2]))
        _qcov.append(b[3])
    elif q in has_any:
        _tier.append("Structure: unknown function")
        _theme.append(None)
        _qcov.append(b[3] if b is not None else None)
    else:
        _tier.append("No structural match")
        _theme.append(None)
        _qcov.append(None)
tab = tab.with_columns(pl.Series("struct_tier", _tier), pl.Series("theme", _theme, dtype=pl.String),
                       pl.Series("struct_qcov", _qcov, dtype=pl.Float64))
tab = tab.with_columns((pl.col("df_recovery") != "no_hit_P_novel").alias("DefenseFinder"))
tab = tab.with_columns(
    ((pl.col("struct_tier") == "Structure: known function") | pl.col("DefenseFinder")).alias("known_function"))
print(tab.group_by("struct_tier").agg(pl.len().alias("n")).sort("n", descending=True))
print(f"\nidentified by an existing method: {tab.filter(pl.col('known_function')).height:,}")
print(f"unresolved by any existing method : {tab.filter(~pl.col('known_function')).height:,}")

parsed 35,157,154 alignments in 180s
proteins with any hit: 101,439   with a confident named hit: 75,787


shape: (3, 2)
┌─────────────────────────────┬───────┐
│ struct_tier                 ┆ n     │
│ ---                         ┆ ---   │
│ str                         ┆ u32   │
╞═════════════════════════════╪═══════╡
│ Structure: known function   ┆ 72393 │
│ Structure: unknown function ┆ 29046 │
│ No structural match         ┆ 7668  │
└─────────────────────────────┴───────┘

identified by an existing method: 78,695
unresolved by any existing method : 30,412


The winning confident structural match of each protein, the source of the structural function names.

In [14]:
winners = pl.DataFrame(
    [(q, b[4], b[5], b[2], b[6], b[1], b[7], b[8], b[3]) for q, b in best.items()],
    schema={"Protein": pl.String, "winner_db": pl.String, "winner_target": pl.String, "winner_description": pl.String,
            "description_from_bfvd_names": pl.Boolean, "winner_tier": pl.Int64, "winner_bits": pl.Float64,
            "winner_evalue": pl.Float64, "winner_qcov": pl.Float64}, orient="row")
winners.write_parquet(FIGT / "structural_match_winner_per_protein.parquet")
print(f"structural_match_winner_per_protein: {winners.height:,} proteins")
print(winners.group_by("winner_db").agg(pl.len().alias("n")).sort("n", descending=True))

structural_match_winner_per_protein: 75,787 proteins
shape: (3, 2)
┌───────────┬───────┐
│ winner_db ┆ n     │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ afdb50    ┆ 67874 │
│ bfvd      ┆ 7434  │
│ pdb       ┆ 479   │
└───────────┴───────┘


## Step 5: Family and protein counts

Two counts are reported with their definitions: unresolved proteins, which no method identifies, and fully unresolved families, in which no member is identified by DefenseFinder or structural search.

Compute both quantities and the family size distribution of the unresolved set.

In [15]:
unres = tab.filter(~pl.col("known_function"))
fam_all = tab.group_by("cluster").agg(pl.len().alias("n"),
                                      (~pl.col("known_function")).all().alias("fully_unresolved"))
fam_novel = fam_all.filter(pl.col("fully_unresolved"))
sizes = unres.group_by("cluster").len().rename({"len": "n"}).sort("n", descending=True)
counts = {
    "unresolved_proteins": unres.height,
    "clusters_containing_unresolved": sizes.height,
    "fully_unresolved_families": fam_novel.height,
    "proteins_in_fully_unresolved_families": int(fam_novel["n"].sum()),
    "largest_unresolved_family": int(sizes["n"].max()),
    "median_unresolved_family_size": float(sizes["n"].median()),
    "singleton_unresolved_families": int((sizes["n"] == 1).sum()),
    "unresolved_families_ge5": int((sizes["n"] >= 5).sum()),
}
for k, v in counts.items():
    print(f"  {k:42s} {v:,}" if isinstance(v, int) else f"  {k:42s} {v}")
pl.DataFrame({"quantity": list(counts.keys()), "value": [float(v) for v in counts.values()]}
             )
sizes

  unresolved_proteins                        30,412
  clusters_containing_unresolved             106
  fully_unresolved_families                  56
  proteins_in_fully_unresolved_families      100
  largest_unresolved_family                  2,706
  median_unresolved_family_size              1.0
  singleton_unresolved_families              57
  unresolved_families_ge5                    48


cluster,n
i64,u32
30,2706
0,2615
11,2576
12,2306
19,1888
7,1696
20,1657
28,1435
15,1331


## Step 6: Assignment depth of the unresolved proteins

The depth at which a propagated label is assigned at the primary precision target, for proteins with a known structure and for proteins no method resolves.

Depth of assignment for the structure-known and unresolved groups.

In [16]:
groups = [("Structure-known", tab.filter(pl.col("struct_tier") == "Structure: known function")),
          ("Unresolved", tab.filter(~pl.col("known_function")))]
rows = []
for gn, g in groups:
    n = g.height
    for lvl in ["specific", "L1", "category", "unassigned"]:
        k = g.filter(pl.col("final_level") == lvl).height
        rows.append({"group": gn, "group_n": n, "level": lvl, "n": k, "pct": 100 * k / n if n else 0.0})
depth = pl.DataFrame(rows)
depth
print(depth.pivot(values="pct", index="level", on="group").fill_null(0))
for gn, g in groups:
    lab = g.filter(pl.col("final_level") != "unassigned").height
    print(f"{gn} (n={g.height:,}): any label {100*lab/g.height:.1f}%, "
          f"specific {100*g.filter(pl.col('final_level')=='specific').height/g.height:.1f}%")

shape: (4, 3)
┌────────────┬─────────────────┬────────────┐
│ level      ┆ Structure-known ┆ Unresolved │
│ ---        ┆ ---             ┆ ---        │
│ str        ┆ f64             ┆ f64        │
╞════════════╪═════════════════╪════════════╡
│ specific   ┆ 43.953145       ┆ 38.928712  │
│ L1         ┆ 20.413576       ┆ 13.978035  │
│ category   ┆ 4.003149        ┆ 6.158753   │
│ unassigned ┆ 31.63013        ┆ 40.9345    │
└────────────┴─────────────────┴────────────┘
Structure-known (n=72,393): any label 68.4%, specific 44.0%
Unresolved (n=30,412): any label 59.1%, specific 38.9%


Broad functional composition of the unresolved proteins, at the precision target.

In [17]:
ur_lab = unres.filter(pl.col("final_level").is_in(["specific", "L1"]))
print(f"unresolved with a label at L1 depth or deeper: {ur_lab.height:,} "
      f"({100*ur_lab.height/unres.height:.1f}% of {unres.height:,})")
comp = (ur_lab.group_by("raw_category").agg(pl.len().alias("n"))
        .with_columns((100 * pl.col("n") / ur_lab.height).round(1).alias("pct"))
        .sort("n", descending=True))
print(comp)
comp
top_l1 = (ur_lab.filter(pl.col("raw_L1").is_not_null()).group_by(["raw_category", "raw_L1"])
          .agg(pl.len().alias("n")).sort(["raw_category", "n"], descending=[False, True]))
top_l1
print()
for c in CAT3:
    print(f"top {c} L1 categories among unresolved:")
    print(top_l1.filter(pl.col("raw_category") == c).select(["raw_L1", "n"]).head(6))

unresolved with a label at L1 depth or deeper: 16,090 (52.9% of 30,412)
shape: (3, 3)
┌───────────────┬───────┬──────┐
│ raw_category  ┆ n     ┆ pct  │
│ ---           ┆ ---   ┆ ---  │
│ str           ┆ u32   ┆ f64  │
╞═══════════════╪═══════╪══════╡
│ regulatory    ┆ 12619 ┆ 78.4 │
│ physiological ┆ 3245  ┆ 20.2 │
│ metabolic     ┆ 226   ┆ 1.4  │
└───────────────┴───────┴──────┘

top metabolic L1 categories among unresolved:
shape: (6, 2)
┌───────────────────────────────┬─────┐
│ raw_L1                        ┆ n   │
│ ---                           ┆ --- │
│ str                           ┆ u32 │
╞═══════════════════════════════╪═════╡
│ Amino acid metabolism         ┆ 78  │
│ Carbohydrate metabolism       ┆ 34  │
│ Other                         ┆ 29  │
│ Lipid & fatty acid metabolism ┆ 26  │
│ Energy metabolism             ┆ 20  │
│ Cofactor & vitamin metabolism ┆ 15  │
└───────────────────────────────┴─────┘
top physiological L1 categories among unresolved:
shape: (5, 2)
┌───────────

## Step 7: Output tables

`novel_master_per_protein.parquet` is read by soil_gut_avgs_figures.Rmd and functional_propagation_figures.Rmd.

UMAP of the sequence-similarity invisible AVGs, with method attribution and assignment depth.

In [18]:
_CAT3L = ["Metabolic", "Physiological", "Regulatory"]
UM = OUT / "hi_umap2d.npy"
if UM.exists() and np.load(UM).shape[0] == tab.height:
    U = np.load(UM)
else:
    import umap
    U = umap.UMAP(n_neighbors=30, min_dist=0.3, metric="cosine", random_state=42,
                  n_components=2).fit_transform(np.load(OUT / "hi_emb.npy"))
    np.save(UM, U.astype(np.float32))
tab = tab.with_columns(pl.Series("umap1", U[:, 0].astype(np.float32)),
                       pl.Series("umap2", U[:, 1].astype(np.float32)))
tab = tab.with_columns(
    pl.when(pl.col("DefenseFinder")).then(pl.lit("Identified by DefenseFinder"))
      .when(pl.col("struct_tier") == "Structure: known function").then(pl.lit("Identified by structure"))
      .otherwise(pl.lit("Identified by CheckAMG de-novo only")).alias("identified_by"),
    pl.when(pl.col("final_level") == "unassigned").then(pl.lit("No reliable label"))
      .otherwise(pl.col("raw_category")).alias("func_category"),
    pl.when(pl.col("DefenseFinder")).then(pl.lit("Physiological"))
      .when((pl.col("struct_tier") == "Structure: known function")
            & pl.col("theme").is_in(_CAT3L)).then(pl.col("theme"))
      .otherwise(None).alias("theme_struct"))
tab.select(["Protein", "umap1", "umap2", "cluster", "ecosystem", "struct_tier", "known_function",
            "identified_by", "func_category", "final_level", "final_label", "raw_L1", "d1",
            "theme", "theme_struct", "raw_category", "DefenseFinder",
            "label_source"]).write_parquet(FIGT / "novel_master_per_protein.parquet")
print(tab.group_by("identified_by").agg(pl.len().alias("n")).sort("n", descending=True))

shape: (3, 2)
┌─────────────────────────────────────┬───────┐
│ identified_by                       ┆ n     │
│ ---                                 ┆ ---   │
│ str                                 ┆ u32   │
╞═════════════════════════════════════╪═══════╡
│ Identified by structure             ┆ 55398 │
│ Identified by CheckAMG de-novo only ┆ 30412 │
│ Identified by DefenseFinder         ┆ 23297 │
└─────────────────────────────────────┴───────┘


Proteins and families per structural tier.

In [19]:
STR = (tab.group_by("struct_tier").agg(pl.len().alias("n_proteins"),
                                       pl.col("cluster").n_unique().alias("n_families"))
       .rename({"struct_tier": "tier"}).sort("n_proteins", descending=True))
print(STR)

shape: (3, 3)
┌─────────────────────────────┬────────────┬────────────┐
│ tier                        ┆ n_proteins ┆ n_families │
│ ---                         ┆ ---        ┆ ---        │
│ str                         ┆ u32        ┆ u32        │
╞═════════════════════════════╪════════════╪════════════╡
│ Structure: known function   ┆ 72393      ┆ 228        │
│ Structure: unknown function ┆ 29046      ┆ 97         │
│ No structural match         ┆ 7668       ┆ 64         │
└─────────────────────────────┴────────────┴────────────┘


## Step 8: Families of the unresolved proteins

Coherence of the 50 embedding families that hold both identified and unresolved proteins, and the MMseqs2 sequence clusters the unresolved proteins span. A family's dominant theme, external category, and DefenseFinder system are the most frequent values among its identified members, with ties broken alphabetically. The DefenseFinder system of a protein is the system of its highest-scoring profile hit that passes the gathering threshold. An unresolved protein's label category is the category of its propagated label at the depth it was assigned.

In [20]:
reference = pl.read_parquet(PCACHE / "labeled_reference.parquet", columns=["ref_specific", "ref_L1", "ref_category"])
specific_to_category = dict(reference.group_by("ref_specific").agg(pl.col("ref_category").mode().sort().first()).iter_rows())
l1_to_category = dict(reference.group_by("ref_L1").agg(pl.col("ref_category").mode().sort().first()).iter_rows())
top_system = (df_hits.filter(pl.col("ga_pass")).sort(["bitscore", "System"], descending=[True, False])
              .unique("Protein", keep="first", maintain_order=True).select(["Protein", "System"]))
members = tab.join(top_system, on="Protein", how="left").with_columns(
    pl.when(pl.col("final_level") == "category").then(pl.col("final_label"))
      .when(pl.col("final_level") == "L1").then(pl.col("final_label").replace_strict(l1_to_category, default=None))
      .when(pl.col("final_level") == "specific").then(pl.col("final_label").replace_strict(specific_to_category, default=None))
      .otherwise(None).alias("label_category"))


def dominant(values):
    values = values.drop_nulls()
    if values.len() == 0:
        return None, None, 0
    counts = values.value_counts().sort(["count", values.name], descending=[True, False])
    return counts[0, 0], round(counts[0, 1] / values.len(), 4), values.len()


family_sizes = members.group_by("cluster").agg(pl.len().alias("n"), pl.col("known_function").sum().alias("n_identified"),
                                               (~pl.col("known_function")).sum().alias("n_unresolved"))
mixed = family_sizes.filter((pl.col("n_identified") > 0) & (pl.col("n_unresolved") > 0)).sort(
    ["n_unresolved", "cluster"], descending=[True, False])
rows = []
for leiden in mixed["cluster"].to_list():
    family = members.filter(pl.col("cluster") == leiden)
    identified = family.filter(pl.col("known_function"))
    labeled = family.filter(~pl.col("known_function") & (pl.col("final_level") != "unassigned"))
    theme, theme_share, n_theme = dominant(identified["theme"])
    external, external_share, n_external = dominant(identified["theme_struct"])
    defense = identified.filter(pl.col("DefenseFinder"))
    system, system_share, _ = dominant(defense["System"])
    matches = int((labeled["label_category"].str.to_lowercase() == external.lower()).sum()) if external else None
    rows.append({"leiden": leiden, "n": family.height, "n_identified": identified.height,
                 "n_unresolved": family.height - identified.height, "n_identified_with_theme": n_theme,
                 "dominant_theme": theme, "dominant_theme_share": theme_share,
                 "n_identified_with_external_category": n_external, "dominant_external_category": external,
                 "dominant_external_category_share": external_share, "n_identified_defensefinder": defense.height,
                 "dominant_df_system": system, "dominant_df_system_share": system_share,
                 "n_unresolved_labeled": labeled.height, "n_unresolved_label_matches_dominant": matches,
                 "unresolved_match_rate": round(matches / labeled.height, 4) if matches is not None and labeled.height else None})
family_coherence = pl.DataFrame(rows, infer_schema_length=None)
family_coherence.write_csv(FIGT / "family_coherence.csv")
print(f"family_coherence: {family_coherence.height} families holding {family_coherence['n'].sum():,} proteins")
print(family_coherence.select(["leiden", "n", "n_unresolved", "dominant_external_category",
                               "dominant_external_category_share", "unresolved_match_rate"]).head(10))

family_coherence: 50 families holding 108,620 proteins
shape: (10, 6)
┌────────┬──────┬──────────────┬──────────────────────┬──────────────────────┬─────────────────────┐
│ leiden ┆ n    ┆ n_unresolved ┆ dominant_external_ca ┆ dominant_external_ca ┆ unresolved_match_ra │
│ ---    ┆ ---  ┆ ---          ┆ tegory               ┆ tegory_share         ┆ te                  │
│ i64    ┆ i64  ┆ i64          ┆ ---                  ┆ ---                  ┆ ---                 │
│        ┆      ┆              ┆ str                  ┆ f64                  ┆ f64                 │
╞════════╪══════╪══════════════╪══════════════════════╪══════════════════════╪═════════════════════╡
│ 30     ┆ 7583 ┆ 2706         ┆ Physiological        ┆ 0.7207               ┆ 0.547               │
│ 0      ┆ 6681 ┆ 2615         ┆ Regulatory           ┆ 0.506                ┆ 0.9879              │
│ 11     ┆ 2939 ┆ 2576         ┆ Metabolic            ┆ 0.5583               ┆ 0.0128              │
│ 12     ┆ 5455 ┆ 230

Sequence clusters spanned by the unresolved proteins in the shared MMseqs2 universe of labeled training proteins and applied AVGs, at 30 and 50 percent identity. The cluster size is the size of the whole universe cluster a protein falls in.

In [21]:
unresolved = members.filter(~pl.col("known_function")).select(["Protein", "cluster"])
summary_rows, span = [], None
for level in ["30", "50"]:
    clusters = pl.read_csv(PCACHE / "seqclust" / f"clu_all_id{level}.tsv", separator="\t", has_header=False,
                           new_columns=["rep", "member"])
    cluster_size = clusters.group_by("rep").len().rename({"len": "cluster_size"})
    placed = unresolved.join(clusters, left_on="Protein", right_on="member").join(cluster_size, on="rep")
    per_cluster = placed.group_by("rep").agg(pl.len().alias("n_residual"), pl.col("cluster_size").first())
    singletons = per_cluster.filter(pl.col("cluster_size") == 1).height
    summary_rows.append({"min_seq_id": f"id{level}", "n": placed.height, "distinct_clusters_spanned": per_cluster.height,
                         "universe_singleton_clusters": singletons,
                         "universe_singleton_share": round(singletons / placed.height, 4),
                         "clusters_holding_exactly_one_residual": per_cluster.filter(pl.col("n_residual") == 1).height,
                         f"residuals_alone_in_their_cluster_among_the_{placed.height}":
                             per_cluster.filter(pl.col("n_residual") == 1).height,
                         "median_universe_cluster_size": placed["cluster_size"].median(),
                         "largest_cluster_share_of_residual": per_cluster["n_residual"].max()})
    if level == "30":
        span = (placed.group_by("cluster")
                .agg(pl.len().alias("n_unresolved"), pl.col("rep").n_unique().alias("id30_clusters"),
                     pl.col("rep").filter(pl.col("cluster_size") == 1).n_unique().alias("id30_universe_singletons"),
                     pl.col("cluster_size").median().alias("median_id30_clusize"))
                .with_columns((pl.col("id30_clusters") / pl.col("n_unresolved")).round(3).alias("clusters_per_protein"))
                .join(family_coherence.select(["leiden", "n", "n_identified", "dominant_external_category"]),
                      left_on="cluster", right_on="leiden", how="left")
                .select([pl.lit("id30").alias("min_seq_id"), pl.col("cluster").alias("leiden"), "n_unresolved",
                         "id30_clusters", "id30_universe_singletons", "median_id30_clusize", "clusters_per_protein",
                         pl.col("n").cast(pl.Float64), pl.col("n_identified").cast(pl.Float64), "dominant_external_category"])
                .sort(["n_unresolved", "leiden"], descending=[True, False]))
span_summary = pl.DataFrame(summary_rows)
span.write_csv(FIGT / "residual_cluster_span.csv")
span_summary.write_csv(FIGT / "residual_cluster_span_summary.csv")
print(span_summary)

shape: (2, 9)
┌────────────┬───────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ min_seq_id ┆ n     ┆ distinct_c ┆ universe_s ┆ … ┆ clusters_ ┆ residuals ┆ median_un ┆ largest_c │
│ ---        ┆ ---   ┆ lusters_sp ┆ ingleton_c ┆   ┆ holding_e ┆ _alone_in ┆ iverse_cl ┆ luster_sh │
│ str        ┆ i64   ┆ anned      ┆ lusters    ┆   ┆ xactly_on ┆ _their_cl ┆ uster_siz ┆ are_of_re │
│            ┆       ┆ ---        ┆ ---        ┆   ┆ e_residua ┆ uster_amo ┆ e         ┆ sidual    │
│            ┆       ┆ i64        ┆ i64        ┆   ┆ l         ┆ ng_the_30 ┆ ---       ┆ ---       │
│            ┆       ┆            ┆            ┆   ┆ ---       ┆ 412       ┆ f64       ┆ i64       │
│            ┆       ┆            ┆            ┆   ┆ i64       ┆ ---       ┆           ┆           │
│            ┆       ┆            ┆            ┆   ┆           ┆ i64       ┆           ┆           │
╞════════════╪═══════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═

## Step 9: CheckAMG-PST against the frozen ESM-2 backbone

Functional label transfer from CheckAMG-PST embeddings and from the frozen ESM-2 (`esm2_t30_150M`) embeddings it was finetuned from, on the held-out AVG-like test proteins, by label depth and by identity to the training set. Both use the deployed 1-NN rule: the nearest AVG-like neighbor among the top 50 donates its category, specific function, and that function's L1. A protein is compared when both embeddings place an AVG-like protein among its top 50 neighbors. The neighbor arrays of both searches are built and cached by functional_propagation.ipynb.

Specific functions are compared after normalization. Micro precision is the fraction of proteins labeled correctly, and macro precision is the unweighted mean over true classes of the fraction labeled correctly. Identity to the training set is rounded to 3 decimals before banding, and proteins with no recorded identity form their own band.

In [22]:
def normalize_function(values):
    out = []
    for s in values:
        s = re.sub(r"\s*\[EC:[^\]]*\]", "", s)
        s = re.sub(r"^[A-Za-z0-9_/]+(,\s*[A-Za-z0-9_/.]+)*;\s*", "", s)
        out.append(re.sub(r"\s+", " ", s).strip().lower())
    return np.array(out, dtype=object)


train_order = pl.read_parquet(OUT / "train_ref_index_order.parquet", columns=["Protein Classification", "Function"])
avg_like = train_order["Protein Classification"].is_in(CAT3).fill_null(False).to_numpy()
specific_to_l1 = dict(zip(reference["ref_specific"].to_list(), reference["ref_L1"].to_list()))
esm_queries = pl.read_parquet(OUT / "esm_query_names.parquet").with_row_index("esm_row")
pst_meta = pl.read_parquet(OUT / "test_avglike_meta.parquet").with_row_index("pst_row")
queries = esm_queries.join(pst_meta.group_by("Protein").agg(pl.col("pst_row").min()), on="Protein", how="inner")


def nearest_avglike(neighbors):
    is_avg = avg_like[neighbors]
    found = is_avg.any(1)
    return found, neighbors[np.arange(neighbors.shape[0]), np.where(found, is_avg.argmax(1), 0)]


esm_found, esm_donor = nearest_avglike(np.load(OUT / "esm_query_I.npy")[queries["esm_row"].to_numpy()])
pst_found, pst_donor = nearest_avglike(np.load(OUT / "test_avglike_I.npy")[queries["pst_row"].to_numpy()])
both = esm_found & pst_found
matched = queries.filter(pl.Series(both)).join(pl.read_parquet(PCACHE / "eval_identity.parquet"), on="Protein", how="left")
print(f"held-out AVG-like proteins compared: {matched.height:,}")

true_function = matched["true_fn"].fill_null("").to_numpy().astype(str)
truth = {"category": matched["true_cat"].fill_null("").to_numpy().astype(str),
         "L1": np.array([specific_to_l1.get(f) or "" for f in true_function], dtype=object),
         "specific": normalize_function(true_function)}
donated = {}
for backbone, donor in [("CheckAMG-PST", pst_donor[both]), ("ESM-2", esm_donor[both])]:
    donor_rows = train_order[donor]
    donor_function = donor_rows["Function"].fill_null("").to_numpy().astype(str)
    donated[backbone] = {"category": donor_rows["Protein Classification"].fill_null("").to_numpy().astype(str),
                         "L1": np.array([specific_to_l1.get(f) or "" for f in donor_function], dtype=object),
                         "specific": normalize_function(donor_function)}


def precision(predicted, true):
    correct = (predicted == true) & (predicted != "") & (true != "")
    classes = np.unique(true[true != ""])
    return float(correct.mean()), float(np.mean([correct[true == c].mean() for c in classes])), len(classes)


identity = matched["max_ident_to_train"].cast(pl.Float64).round(3).to_numpy()
bands = [("no recorded identity", None, None, np.isnan(identity)),
         ("[0.00, 0.30)", 0.0, 0.3, identity < 0.3), ("[0.30, 0.50)", 0.3, 0.5, (identity >= 0.3) & (identity < 0.5)),
         ("[0.50, 0.90)", 0.5, 0.9, (identity >= 0.5) & (identity < 0.9)), ("[0.90, 1.00]", 0.9, None, identity >= 0.9),
         ("pooled (all matched)", None, None, np.ones(len(identity), bool))]
band_rows, depth_rows = [], []
for band, low, high, in_band in bands:
    for depth in ["category", "L1", "specific"]:
        evaluable = in_band & (truth["L1"] != "") if depth == "L1" else in_band
        result = {b: precision(donated[b][depth][evaluable], truth[depth][evaluable]) for b in donated}
        n = int(evaluable.sum())
        base = {"band": band, "band_lo": low, "band_hi": high, "depth": depth, "n_truth_classes": result["ESM-2"][2]}
        for backbone in ["CheckAMG-PST", "ESM-2"]:
            band_rows += [{**base, "embedding": backbone, "metric": "micro_precision", "value": result[backbone][0]},
                          {**base, "embedding": backbone, "metric": "macro_precision", "value": result[backbone][1]},
                          {**base, "embedding": backbone, "metric": "n", "value": float(n)}]
        gap = "ESM-2 minus CheckAMG-PST"
        band_rows += [{**base, "embedding": gap, "metric": "micro_precision_gap", "value": result["ESM-2"][0] - result["CheckAMG-PST"][0]},
                      {**base, "embedding": gap, "metric": "macro_precision_gap", "value": result["ESM-2"][1] - result["CheckAMG-PST"][1]},
                      {**base, "embedding": gap, "metric": "n", "value": float(n)}]
        if band.startswith("pooled"):
            for backbone, name in [("CheckAMG-PST", "CheckAMG-PST"), ("ESM-2", "ESM-2 backbone")]:
                depth_rows.append({"rule": "nearest_avglike_1nn", "level": depth, "backbone": name, "n_matched_evaluable": n,
                                   "n_true_classes": result[backbone][2], "micro_precision": result[backbone][0],
                                   "macro_precision": result[backbone][1]})
backbone_by_band = pl.DataFrame(band_rows, schema_overrides={"band_lo": pl.Float64, "band_hi": pl.Float64})
backbone_by_depth = pl.DataFrame(depth_rows)
backbone_by_band.write_csv(FIGT / "backbone_ablation_by_identity_band.csv")
backbone_by_depth.write_csv(FIGT / "backbone_ablation_by_depth.csv")
print(backbone_by_depth)

held-out AVG-like proteins compared: 133,086


shape: (6, 7)
┌──────────────┬──────────┬──────────────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ rule         ┆ level    ┆ backbone     ┆ n_matched_ev ┆ n_true_class ┆ micro_preci ┆ macro_preci │
│ ---          ┆ ---      ┆ ---          ┆ aluable      ┆ es           ┆ sion        ┆ sion        │
│ str          ┆ str      ┆ str          ┆ ---          ┆ ---          ┆ ---         ┆ ---         │
│              ┆          ┆              ┆ i64          ┆ i64          ┆ f64         ┆ f64         │
╞══════════════╪══════════╪══════════════╪══════════════╪══════════════╪═════════════╪═════════════╡
│ nearest_avgl ┆ category ┆ CheckAMG-PST ┆ 133086       ┆ 3            ┆ 0.895556    ┆ 0.852418    │
│ ike_1nn      ┆          ┆              ┆              ┆              ┆             ┆             │
│ nearest_avgl ┆ category ┆ ESM-2        ┆ 133086       ┆ 3            ┆ 0.988233    ┆ 0.983925    │
│ ike_1nn      ┆          ┆ backbone     ┆              ┆              ┆     